In [1]:
import numpy as np
import gymnasium as gym

import warnings ; warnings.filterwarnings('ignore')

from pprint import pprint
from tqdm import tqdm_notebook as tqdm

from itertools import cycle

import random

np.set_printoptions(suppress=True)
random.seed(123); np.random.seed(123)

In [2]:
# P = gym.make('FrozenLake-v1').env.unwrapped.P

env = gym.make('FrozenLake-v1')
P = env.env.unwrapped.P
init_state = env.reset()
goal_state = 6

- The outer dictionary keys are the states
- The inner dictionary keys are the actions.
- The value of the inner dictionary is a list with all possible transitions for that state-action pair
- The transition tuples have four values:
  - the probability of that transition,
  - the next state,
  - the reward, and a flag indicating
  - whether the next state is terminal.

In [3]:
P

{0: {0: [(0.33333333333333337, 0, 0, False),
   (0.3333333333333333, 0, 0, False),
   (0.33333333333333337, 4, 0, False)],
  1: [(0.33333333333333337, 0, 0, False),
   (0.3333333333333333, 4, 0, False),
   (0.33333333333333337, 1, 0, False)],
  2: [(0.33333333333333337, 4, 0, False),
   (0.3333333333333333, 1, 0, False),
   (0.33333333333333337, 0, 0, False)],
  3: [(0.33333333333333337, 1, 0, False),
   (0.3333333333333333, 0, 0, False),
   (0.33333333333333337, 0, 0, False)]},
 1: {0: [(0.33333333333333337, 1, 0, False),
   (0.3333333333333333, 0, 0, False),
   (0.33333333333333337, 5, 0, True)],
  1: [(0.33333333333333337, 0, 0, False),
   (0.3333333333333333, 5, 0, True),
   (0.33333333333333337, 2, 0, False)],
  2: [(0.33333333333333337, 5, 0, True),
   (0.3333333333333333, 2, 0, False),
   (0.33333333333333337, 1, 0, False)],
  3: [(0.33333333333333337, 2, 0, False),
   (0.3333333333333333, 1, 0, False),
   (0.33333333333333337, 0, 0, False)]},
 2: {0: [(0.33333333333333337, 2, 0

In [4]:
def print_policy(pi, P, action_symbols=('<', '>','v', '^'), n_cols=4, title='Policy:'):
    print(title)
    arrs = {k:v for k,v in enumerate(action_symbols)}
    for s in range(len(P)):
        a = pi(s)
        print("| ", end="")
        if np.all([done for action in P[s].values() for _, _, _, done in action]):
            print("".rjust(9), end=" ")
        else:
            print(str(s).zfill(2), arrs[a].rjust(6), end=" ")
        if (s + 1) % n_cols == 0: print("|")

In [5]:
init_state = env.reset()
goal_state = 15

LEFT, RIGHT, DOWN, UP = range(4)

## Policy

pi = lambda s: {
    0:LEFT, 1:LEFT, 2:LEFT, 3:LEFT,
    4:LEFT, 5:LEFT, 6:LEFT, 7:LEFT,
    8:LEFT, 9:LEFT, 10:LEFT, 11:LEFT,
    12:LEFT, 13:RIGHT, 14:RIGHT, 15:LEFT
}[s]


print_policy(pi, P, action_symbols=('<', '>','v', '^'), n_cols=4)


Policy:
| 00      < | 01      < | 02      < | 03      < |
| 04      < |           | 06      < |           |
| 08      < | 09      < | 10      < |           |
|           | 13      > | 14      > |           |


In [6]:
n = [1, 2, 3, 4, 5, 6]
even = lambda x: x % 2 == 0, n
print(list(even[1]))

[1, 2, 3, 4, 5, 6]


In [7]:
len(P) ## number of states

16

## policy evaluation

In [8]:
def policy_eval(pi, P, gamma=1.0, theta=1e-10):
    prev_V = np.zeros(len(P), dtype=np.float64)
    while True:
        V = np.zeros(len(P), dtype=np.float64)
        for s in range(len(P)):
            for prob, next_state, reward, done in P[s][pi(s)]:
                V[s] += prob * (reward + gamma * prev_V[next_state] * (not done))
        if np.max(np.abs(prev_V - V)) < theta:
            break
        prev_V = V.copy()
    return V

In [9]:
V=policy_eval(pi, P)
V

array([0.        , 0.        , 0.05982906, 0.02991453, 0.        ,
       0.        , 0.11965812, 0.        , 0.        , 0.11111111,
       0.2991453 , 0.        , 0.        , 0.33333333, 0.66666667,
       0.        ])

In [10]:
## simple data visualization

def print_state_value_function(V, P, n_cols=4, prec=3, title='State-value function:'):
    print(title)
    for s in range(len(P)):
        v = V[s]
        print("| ", end="")
        if np.all([done for action in P[s].values() for _, _, _, done in action]):
            print("".rjust(9), end=" ")
        else:
            print(str(s).zfill(2), '{}'.format(np.round(v, prec)).rjust(6), end=" ")
        if (s + 1) % n_cols == 0: print("|")

print_state_value_function(V, P, n_cols=4)

State-value function:
| 00    0.0 | 01    0.0 | 02   0.06 | 03   0.03 |
| 04    0.0 |           | 06   0.12 |           |
| 08    0.0 | 09  0.111 | 10  0.299 |           |
|           | 13  0.333 | 14  0.667 |           |


### We can compare policies and can rank them by State-value function of the START State

## Policy improvement

In [11]:
def policy_improve(V, P, gamma=1.0):
    Q = np.zeros((len(P), len(P[0])), dtype=np.float64)
    for s in range(len(P)):
        for a in range(len(P[s])):
            for prob, next_state, reward, done in P[s][a]:
                Q[s][a] += prob * (reward + gamma * V[next_state] * (not done))
    new_pi = lambda s: {s:a for s, a in enumerate(np.argmax(Q, axis=1))}[s]
    return new_pi

improved_pi = policy_improve(V, P)

In [12]:
improved_V = policy_eval(improved_pi, P)
print_policy(pi, P, action_symbols=('<', '>','v', '^'), n_cols=4)
print("===============================================\nafter update")
print_policy(improved_pi, P, action_symbols=('<', '>','v', '^'), n_cols=4)
print("===============================================")
print_state_value_function(improved_V, P, n_cols=4, prec=3)


Policy:
| 00      < | 01      < | 02      < | 03      < |
| 04      < |           | 06      < |           |
| 08      < | 09      < | 10      < |           |
|           | 13      > | 14      > |           |
after update
Policy:
| 00      < | 01      > | 02      v | 03      ^ |
| 04      < |           | 06      < |           |
| 08      > | 09      > | 10      < |           |
|           | 13      v | 14      > |           |
State-value function:
| 00  0.231 | 01  0.162 | 02  0.256 | 03  0.256 |
| 04  0.231 |           | 06  0.256 |           |
| 08  0.231 | 09  0.462 | 10  0.513 |           |
|           | 13  0.641 | 14  0.821 |           |


## Policy Iteration
getting the optimal policy

In [13]:
def policy_iteration(P, gamma=1.0, theta=1e-10):

    random_actions = np.random.choice(tuple(P[0].keys()), len(P))

    pi = lambda s: {s:a for s, a in enumerate(random_actions)}[s]

    while True:
        old_pi = {s:pi(s) for s in range(len(P))}
        V = policy_eval(pi, P, gamma, theta)
        pi = policy_improve(V, P, gamma)
        ## checking if the old policy is different from the updated one
        if old_pi == {s:pi(s) for s in range(len(P))}:
            break
    return V, pi


p_optimal_value, p_optimal_policy = policy_iteration(P)

In [14]:
print_policy(pi, P, action_symbols=('<', '>','v', '^'), n_cols=4)
print("===============================================\nafter update")
print_policy(improved_pi, P, action_symbols=('<', '>','v', '^'), n_cols=4)
print("===============================================\nafter update")
print('Optimal policy and state-value function (PI):')

print_policy(p_optimal_policy, P, action_symbols=('<', '>','v', '^'), n_cols=4)
print()
print_state_value_function(p_optimal_value, P, n_cols=4, prec=5)

Policy:
| 00      < | 01      < | 02      < | 03      < |
| 04      < |           | 06      < |           |
| 08      < | 09      < | 10      < |           |
|           | 13      > | 14      > |           |
after update
Policy:
| 00      < | 01      > | 02      v | 03      ^ |
| 04      < |           | 06      < |           |
| 08      > | 09      > | 10      < |           |
|           | 13      v | 14      > |           |
after update
Optimal policy and state-value function (PI):
Policy:
| 00      < | 01      ^ | 02      ^ | 03      ^ |
| 04      < |           | 06      < |           |
| 08      ^ | 09      > | 10      < |           |
|           | 13      v | 14      > |           |

State-value function:
| 00 0.82353 | 01 0.82353 | 02 0.82353 | 03 0.82353 |
| 04 0.82353 |           | 06 0.52941 |           |
| 08 0.82353 | 09 0.82353 | 10 0.76471 |           |
|           | 13 0.88235 | 14 0.94118 |           |


## Value Iteration
getting the optimal policy

In [15]:
def value_iteration(P, gamma=1.0, theta=1e-10):
    V = np.zeros(len(P), dtype=np.float64)
    while True:
        Q = np.zeros((len(P), len(P[0])), dtype=np.float64)
        for s in range(len(P)):
            for a in range(len(P[s])):
                for prob, next_state, reward, done in P[s][a]:
                    Q[s][a] += prob * (reward + gamma * V[next_state] * (not done))
        if np.max(np.abs(V - np.max(Q, axis=1))) < theta:
            break
        V = np.max(Q, axis=1)
    pi = lambda s: {s:a for s, a in enumerate(np.argmax(Q, axis=1))}[s]
    return V, pi

In [16]:
v_optimal_value, v_optimal_policy = value_iteration(P)

In [17]:
print('PI Optimal policy and state-value function:')
print_policy(p_optimal_policy, P, action_symbols=('<', '>','v', '^'), n_cols=4)
print("===============================================")
print_state_value_function(p_optimal_value, P, n_cols=4, prec=5)
print("===============================================")

print("===============================================")
print('VI Optimal policy and state-value function:')
print_policy(v_optimal_policy, P, action_symbols=('<', '>','v', '^'), n_cols=4)
print("===============================================")
print_state_value_function(v_optimal_value, P, n_cols=4, prec=5)

PI Optimal policy and state-value function:
Policy:
| 00      < | 01      ^ | 02      ^ | 03      ^ |
| 04      < |           | 06      < |           |
| 08      ^ | 09      > | 10      < |           |
|           | 13      v | 14      > |           |
State-value function:
| 00 0.82353 | 01 0.82353 | 02 0.82353 | 03 0.82353 |
| 04 0.82353 |           | 06 0.52941 |           |
| 08 0.82353 | 09 0.82353 | 10 0.76471 |           |
|           | 13 0.88235 | 14 0.94118 |           |
VI Optimal policy and state-value function:
Policy:
| 00      < | 01      ^ | 02      ^ | 03      ^ |
| 04      < |           | 06      < |           |
| 08      ^ | 09      > | 10      < |           |
|           | 13      v | 14      > |           |
State-value function:
| 00 0.82353 | 01 0.82353 | 02 0.82353 | 03 0.82353 |
| 04 0.82353 |           | 06 0.52941 |           |
| 08 0.82353 | 09 0.82353 | 10 0.76471 |           |
|           | 13 0.88235 | 14 0.94118 |           |
